# Heart Disease (s6e2) — First-Order Optimizer Comparison Study

**Furkan Yakkan** — Department of Electrical and Computer Engineering, **Abdullah Gul University (AGU)**, Kayseri, Türkiye.

*ECE 567 — Foundations of Optimization for Machine Learning. Instructor: Dr. Khaled Hejja.*

---

This public Kaggle notebook is the companion to my ECE 567 Final Project, a survey-style paper comparing seven hand-rolled first-order optimizers — **SGD, Polyak Momentum, Nesterov (NAG), AdaGrad, RMSprop, Adam, AdamW** — on the *Playground Series S6E2* heart-disease classification task. The notebook reproduces every key experiment end-to-end: exploratory data analysis, preprocessing, the convex (linear) and non-convex (shallow MLP) optimizer comparisons, a Lasso feature-selection path, and the final Kaggle submission produced by the winning model.

Evaluation metric: **ROC-AUC** (probability submissions).  
Offline best held-out validation AUC: **0.95278** (MLP + Adam).  
Kaggle late-submission private AUC: **0.95288** (MLP + Adam).

## Outline
1. Load data + EDA
2. Preprocessing (standardize continuous, one-hot nominal)
3. Linear baselines (sklearn L-BFGS + hand-rolled vanilla GD)
4. **Seven-optimizer comparison on logistic regression** (convex)
5. **Lasso path** — feature death order vs EDA correlations
6. **Seven-optimizer comparison on shallow MLP** (non-convex)
7. Final retrain on full 630k and submission

## Acknowledgment
This work was carried out as part of the ECE 567 graduate course at **Abdullah Gul University (AGU)**, Kayseri, Türkiye, under the supervision of **Dr. Khaled Hejja**. All code was written from scratch in NumPy; sklearn was used only as a reference solver for benchmarking purposes.

In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

SEED = 2540
np.random.seed(SEED)

DATA_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train.csv' in files and 'test.csv' in files:
        DATA_DIR = root
        break
if DATA_DIR is None:
    raise FileNotFoundError('Attach the playground-series-s6e2 competition via right panel -> Add Input.')
print('DATA_DIR:', DATA_DIR)

train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
print('train', train.shape, '| test', test.shape)
train.head()

## 1. EDA

Map target `Heart Disease` from `Presence/Absence` to `1/0` and look at class balance + correlations.

In [ ]:
TARGET = 'Heart Disease'
train[TARGET] = train[TARGET].map({'Presence': 1, 'Absence': 0})
feature_cols = [c for c in train.columns if c not in ('id', TARGET)]

print('class balance:')
print(train[TARGET].value_counts(normalize=True).round(4).to_string())
print('missing values (train):', int(train.isna().sum().sum()), ' (test):', int(test.isna().sum().sum()))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
train[TARGET].value_counts(normalize=True).sort_index().plot(
    kind='bar', ax=axes[0], color=['steelblue', 'crimson'])
axes[0].set_xticklabels(['Absence (0)', 'Presence (1)'], rotation=0)
axes[0].set_title('Class balance (train)')
axes[0].set_ylabel('proportion')

corr_y = train[feature_cols + [TARGET]].corr()[TARGET].drop(TARGET).sort_values()
corr_y.plot(kind='barh', ax=axes[1], color='purple')
axes[1].set_title('Pearson correlation with Heart Disease')
axes[1].axvline(0, color='black', lw=0.5)
plt.tight_layout(); plt.show()
print('\ntop +corr:', corr_y.tail(3).round(3).to_dict())
print('top -corr:', corr_y.head(3).round(3).to_dict())

## 2. Preprocessing

- **Standardize** continuous features (`Age, BP, Cholesterol, Max HR, ST depression, Slope of ST, Number of vessels fluro`).
- **One-hot** truly nominal categoricals (`Chest pain type 1-4, EKG results 0-2, Thallium 3/6/7`) — treating these as ordinals would falsely impose monotonicity.
- **Passthrough** binaries (`Sex, FBS over 120, Exercise angina`).

Result: **17 features**. Fit the preprocessor on train and apply identically to test.

In [ ]:
continuous = ['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression',
              'Slope of ST', 'Number of vessels fluro']
nominal    = ['Chest pain type', 'EKG results', 'Thallium']
binary     = ['Sex', 'FBS over 120', 'Exercise angina']

preproc = ColumnTransformer([
    ('cont', StandardScaler(), continuous),
    ('nom',  OneHotEncoder(drop='first', sparse_output=False), nominal),
    ('bin',  'passthrough', binary),
])
X_full = preproc.fit_transform(train[feature_cols]).astype(np.float64)
X_test = preproc.transform(test[feature_cols]).astype(np.float64)
y_full = train[TARGET].values.astype(np.float64)
test_ids = test['id'].values
ohe_names = preproc.named_transformers_['nom'].get_feature_names_out(nominal).tolist()
feat_names = continuous + ohe_names + binary

X_tr, X_val, y_tr, y_val = train_test_split(
    X_full, y_full, test_size=0.2, random_state=SEED, stratify=y_full)
print('post-preproc:', X_full.shape, '| features:', len(feat_names))
print('split: train', X_tr.shape, 'val', X_val.shape)

## 3. Linear baselines

`sklearn` L-BFGS for the convex reference optimum, and a hand-rolled vanilla full-batch GD that explicitly writes out the cross-entropy gradient.

In [ ]:
def sigmoid(z):
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos]); out[~pos] = ez / (1.0 + ez)
    return out

def bce_loss(y, p, eps=1e-12):
    p = np.clip(p, eps, 1-eps)
    return -np.mean(y*np.log(p) + (1-y)*np.log(1-p))

# sklearn L-BFGS
t0 = time.time()
sk = LogisticRegression(penalty='l2', C=1e8, solver='lbfgs', max_iter=2000, n_jobs=-1).fit(X_tr, y_tr)
LBFGS_AUC = roc_auc_score(y_val, sk.predict_proba(X_val)[:, 1])
print(f'sklearn L-BFGS    val AUC = {LBFGS_AUC:.5f}   t={time.time()-t0:.1f}s   iter={sk.n_iter_[0]}')

# hand-rolled vanilla full-batch GD
def grad_logreg(X, y, w, b):
    p = sigmoid(X @ w + b)
    return X.T @ (p - y) / X.shape[0], (p - y).mean()

t0 = time.time()
w = np.zeros(X_tr.shape[1]); b = 0.0
for k in range(500):
    gw, gb = grad_logreg(X_tr, y_tr, w, b)
    w -= 0.5 * gw; b -= 0.5 * gb
GD_AUC = roc_auc_score(y_val, sigmoid(X_val @ w + b))
print(f'vanilla GD (500)  val AUC = {GD_AUC:.5f}   t={time.time()-t0:.1f}s')

## 4. Seven optimizers on logistic regression (convex)

All sharing one mini-batch loop. Update rules:
- **SGD**: $\theta \leftarrow \theta - \eta g$
- **Momentum**: $v \leftarrow \mu v + g,\ \theta \leftarrow \theta - \eta v$
- **NAG**: Sutskever look-ahead form
- **AdaGrad**: $G \leftarrow G + g^2,\ \theta \leftarrow \theta - \eta g/(\sqrt G + \epsilon)$
- **RMSprop**: $E \leftarrow \beta E + (1-\beta) g^2,\ \theta \leftarrow \theta - \eta g/(\sqrt E + \epsilon)$
- **Adam**: bias-corrected first + second moments
- **AdamW**: Adam + decoupled L2

Best lrs (taken from the offline grid sweep in the course paper).

In [ ]:
class SGD:
    def __init__(self, shape): pass
    def step(self, g, lr): return -lr * g
class Momentum:
    def __init__(self, shape, mu=0.9): self.mu=mu; self.v=np.zeros(shape)
    def step(self, g, lr): self.v = self.mu*self.v + g; return -lr*self.v
class NAG:
    def __init__(self, shape, mu=0.9): self.mu=mu; self.v=np.zeros(shape)
    def step(self, g, lr):
        v_new = self.mu*self.v - lr*g
        d = self.mu*v_new - lr*g
        self.v = v_new; return d
class AdaGrad:
    def __init__(self, shape, eps=1e-8): self.eps=eps; self.G=np.zeros(shape)
    def step(self, g, lr): self.G = self.G + g*g; return -lr*g/(np.sqrt(self.G)+self.eps)
class RMSprop:
    def __init__(self, shape, beta=0.9, eps=1e-8): self.beta=beta; self.eps=eps; self.E=np.zeros(shape)
    def step(self, g, lr):
        self.E = self.beta*self.E + (1-self.beta)*g*g
        return -lr*g/(np.sqrt(self.E)+self.eps)
class Adam:
    def __init__(self, shape, b1=0.9, b2=0.999, eps=1e-8):
        self.b1=b1; self.b2=b2; self.eps=eps
        self.m=np.zeros(shape); self.v=np.zeros(shape); self.t=0
    def step(self, g, lr):
        self.t += 1
        self.m = self.b1*self.m + (1-self.b1)*g
        self.v = self.b2*self.v + (1-self.b2)*g*g
        mh = self.m/(1-self.b1**self.t); vh = self.v/(1-self.b2**self.t)
        return -lr*mh/(np.sqrt(vh)+self.eps)
class AdamW(Adam):
    def __init__(self, shape, wd=1e-4, **kw): super().__init__(shape, **kw); self.wd=wd

BEST_LR_LIN = {'SGD':1.0, 'Momentum':0.1, 'NAG':0.1,
                'AdaGrad':0.5, 'RMSprop':0.005, 'Adam':0.01, 'AdamW':0.01}
OPT_CLASSES = {'SGD':SGD, 'Momentum':Momentum, 'NAG':NAG,
                'AdaGrad':AdaGrad, 'RMSprop':RMSprop, 'Adam':Adam, 'AdamW':AdamW}

def train_logreg(opt_cls, lr, X, y, X_val, y_val, epochs=15, batch=2048, seed=SEED):
    rng = np.random.default_rng(seed)
    D = X.shape[1]; w = np.zeros(D); b = 0.0
    opt_w = opt_cls(D) if opt_cls is not AdamW else opt_cls(D, wd=1e-4)
    opt_b = opt_cls(()) if opt_cls is not AdamW else opt_cls((), wd=1e-4)
    hist = []
    N = X.shape[0]; nb = int(np.ceil(N/batch))
    for ep in range(epochs):
        idx = rng.permutation(N)
        for bi in range(nb):
            bb = idx[bi*batch:(bi+1)*batch]
            gw, gb = grad_logreg(X[bb], y[bb], w, b)
            dw = opt_w.step(gw, lr); db = float(opt_b.step(np.array(gb), lr))
            if opt_cls is AdamW:
                dw = dw - lr*opt_w.wd*w
            w += dw; b += db
        hist.append(roc_auc_score(y_val, sigmoid(X_val @ w + b)))
    return w, b, hist

In [ ]:
lin_curves = {}
for name, cls in OPT_CLASSES.items():
    t0 = time.time()
    _, _, hist = train_logreg(cls, BEST_LR_LIN[name], X_tr, y_tr, X_val, y_val)
    lin_curves[name] = hist
    print(f'{name:9s}  lr={BEST_LR_LIN[name]:<7g}  best AUC={max(hist):.5f}  '
          f'final={hist[-1]:.5f}  t={time.time()-t0:.1f}s')

fig, ax = plt.subplots(figsize=(8, 4.5))
colors = plt.cm.tab10(np.linspace(0, 1, len(lin_curves)))
for (n, h), c in zip(lin_curves.items(), colors):
    ax.plot(range(1, len(h)+1), h, label=f'{n} lr={BEST_LR_LIN[n]:g}', color=c)
ax.axhline(LBFGS_AUC, ls='--', color='black', alpha=0.6, label=f'L-BFGS ref = {LBFGS_AUC:.4f}')
ax.set_xlabel('epoch'); ax.set_ylabel('val AUC')
ax.set_title('Logistic regression — 7 optimizers (convex)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Reading the figure.** On a convex problem all seven optimizers converge to within 0.00002 AUC of L-BFGS — the differentiation is in *speed*, not final solution. AdaGrad's per-coordinate scaling handles the sparse one-hot dummies fastest. Adam/AdamW are slow to start because bias correction shrinks their effective step size in early iterations.

## 5. Lasso path

Sweep L1 strength over a log-scaled grid and watch features die. The features with near-zero target correlation in EDA (`BP, FBS over 120, Cholesterol`) should be killed first.

In [ ]:
Cs = np.logspace(-4, 2, 13)
lasso_rows = []; coef_path = []
for C in Cs:
    lr = LogisticRegression(penalty='l1', solver='saga', C=C,
                            max_iter=2000, tol=1e-3, n_jobs=-1).fit(X_tr, y_tr)
    w = lr.coef_.ravel()
    lasso_rows.append({'C': C, 'n_active': int((np.abs(w) > 1e-8).sum()),
                        'val_auc': roc_auc_score(y_val, lr.predict_proba(X_val)[:, 1])})
    coef_path.append(w)
lasso_df = pd.DataFrame(lasso_rows)
coef_path = np.array(coef_path)
lam = 1.0 / (X_tr.shape[0] * lasso_df['C'].values)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.5))
cmap = plt.cm.tab20(np.linspace(0, 1, coef_path.shape[1]))
for j, name in enumerate(feat_names):
    axes[0].plot(lam, coef_path[:, j], color=cmap[j], label=name, lw=1.4)
axes[0].set_xscale('log'); axes[0].axhline(0, color='black', lw=0.5)
axes[0].set_xlabel(r'$\lambda_1$ (per-sample L1)')
axes[0].set_ylabel('coefficient value'); axes[0].set_title('Lasso coefficient path')
axes[0].grid(alpha=0.3); axes[0].legend(fontsize=7, loc='upper left', bbox_to_anchor=(1, 1))

ax_auc = axes[1]; ax_act = ax_auc.twinx()
ax_auc.plot(lam, lasso_df['val_auc'], 'o-', color='steelblue', label='val AUC')
ax_act.plot(lam, lasso_df['n_active'], 's--', color='crimson', label='# active features')
ax_auc.axhline(LBFGS_AUC, ls=':', color='black', alpha=0.6, label=f'L-BFGS unreg = {LBFGS_AUC:.4f}')
ax_auc.set_xscale('log'); ax_auc.set_xlabel(r'$\lambda_1$')
ax_auc.set_ylabel('val AUC', color='steelblue')
ax_act.set_ylabel('# active features', color='crimson')
ax_auc.set_title('Sparsity vs accuracy'); ax_auc.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Seven optimizers on a shallow MLP (non-convex)

Architecture: **17 inputs → 64 ReLU → 1 sigmoid** with He initialization. Cross-entropy loss. Hand-rolled forward + backward; same optimizer classes from §4 (one instance per parameter tensor).

In [ ]:
def mlp_init(D, H, seed):
    rng = np.random.default_rng(seed)
    return (rng.standard_normal((D,H))*np.sqrt(2/D), np.zeros(H),
            rng.standard_normal(H)*np.sqrt(2/H), 0.0)

def mlp_forward(X, W1, b1, W2, b2):
    pre = X @ W1 + b1; h = np.maximum(pre, 0)
    return pre, h, h @ W2 + b2

def mlp_grads(X, y, W1, b1, W2, b2):
    N = X.shape[0]
    pre, h, z = mlp_forward(X, W1, b1, W2, b2)
    dz = (sigmoid(z) - y) / N
    gW2 = h.T @ dz; gb2 = dz.sum()
    dh = np.outer(dz, W2); dh[pre <= 0] = 0
    return X.T @ dh, dh.sum(axis=0), gW2, gb2

def mlp_predict(X, W1, b1, W2, b2):
    _, _, z = mlp_forward(X, W1, b1, W2, b2)
    return sigmoid(z)

BEST_LR_MLP = {'SGD':1.0, 'Momentum':0.1, 'NAG':0.1,
                'AdaGrad':0.1, 'RMSprop':0.005, 'Adam':0.01, 'AdamW':0.01}

def train_mlp(opt_cls, lr, X, y, X_val, y_val, H=64, epochs=12, batch=2048, seed=SEED):
    D = X.shape[1]
    W1, b1, W2, b2 = mlp_init(D, H, seed)
    def mk(shape): return opt_cls(shape) if opt_cls is not AdamW else opt_cls(shape, wd=1e-4)
    oW1, ob1, oW2, ob2 = mk(W1.shape), mk(b1.shape), mk(W2.shape), mk(())
    rng = np.random.default_rng(seed+1)
    N = X.shape[0]; nb = int(np.ceil(N/batch))
    hist = []
    for ep in range(epochs):
        idx = rng.permutation(N)
        for bi in range(nb):
            bb = idx[bi*batch:(bi+1)*batch]
            gW1, gb1, gW2, gb2 = mlp_grads(X[bb], y[bb], W1, b1, W2, b2)
            dW1 = oW1.step(gW1, lr); db1 = ob1.step(gb1, lr)
            dW2 = oW2.step(gW2, lr); db2 = float(ob2.step(np.array(gb2), lr))
            if opt_cls is AdamW:
                dW1 -= lr*oW1.wd*W1; dW2 -= lr*oW2.wd*W2
            W1 += dW1; b1 += db1; W2 += dW2; b2 += db2
        hist.append(roc_auc_score(y_val, mlp_predict(X_val, W1, b1, W2, b2)))
    return (W1, b1, W2, b2), hist

In [ ]:
mlp_curves = {}
for name, cls in OPT_CLASSES.items():
    t0 = time.time()
    _, hist = train_mlp(cls, BEST_LR_MLP[name], X_tr, y_tr, X_val, y_val)
    mlp_curves[name] = hist
    print(f'{name:9s}  lr={BEST_LR_MLP[name]:<7g}  best AUC={max(hist):.5f}  '
          f'final={hist[-1]:.5f}  t={time.time()-t0:.1f}s')

fig, ax = plt.subplots(figsize=(8, 4.5))
colors = plt.cm.tab10(np.linspace(0, 1, len(mlp_curves)))
for (n, h), c in zip(mlp_curves.items(), colors):
    ax.plot(range(1, len(h)+1), h, label=f'{n} lr={BEST_LR_MLP[n]:g}', color=c)
ax.axhline(LBFGS_AUC, ls='--', color='black', alpha=0.6, label=f'L-BFGS linear = {LBFGS_AUC:.4f}')
ax.set_xlabel('epoch'); ax.set_ylabel('val AUC')
ax.set_title('Shallow MLP — 7 optimizers (non-convex)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Reading the figure.** Once the loss surface becomes non-convex, the optimizers genuinely separate. Adam / AdamW pull ahead, AdaGrad and RMSprop follow, and vanilla SGD lags. All seven still beat the linear L-BFGS baseline, justifying the added model capacity.

## 7. Final retrain on full 630k and Kaggle submission

Retrain the winning MLP (Adam, lr=0.01) on all available labeled data and predict probabilities on the 270k test set.

In [ ]:
t0 = time.time()
(W1, b1, W2, b2), hist_full = train_mlp(Adam, 0.01, X_full, y_full, X_val, y_val,
                                          epochs=15)
print(f'full-data MLP train time: {time.time()-t0:.1f}s')
p_test = mlp_predict(X_test, W1, b1, W2, b2)

sub = pd.DataFrame({'id': test_ids.astype(int), 'Heart Disease': p_test})
sub.to_csv('submission.csv', index=False)
print('wrote submission.csv  rows =', len(sub))
print('proba mean =', round(p_test.mean(), 4), '  median =', round(np.median(p_test), 4))
sub.head()

## Summary of Kaggle scores (from the offline full study)

| Submission | Public AUC | Private AUC |
|---|---|---|
| Shallow MLP + Adam | 0.95119 | **0.95288** |
| 50/50 ensemble (MLP + LR) | 0.95112 | 0.95283 |
| L-BFGS logistic regression | 0.95041 | 0.95213 |

Best held-out val AUC matches private AUC within 0.0001 — the validation split was an honest predictor of test performance.

---

## Acknowledgment

I gratefully acknowledge **Abdullah Gul University (AGU)**, Kayseri, Türkiye, and **Dr. Khaled Hejja** for the ECE 567 course that motivated this study. The Playground Series competition organizers (Yao Yan, Walter Reade, Elizabeth Park, Kaggle) provided the dataset.

**Citation.** Yao Yan, Walter Reade, Elizabeth Park. *Predicting Heart Disease.* https://kaggle.com/competitions/playground-series-s6e2, 2026.